In [1]:
!pip install -r requirements.txt

Processing /wheels/flash_attn-2.6.3-cp310-cp310-linux_x86_64.whl (from -r requirements.txt (line 39))
flash_attn is already installed with the same version as the provided wheel. Use --force-reinstall to force an installation of the wheel.


In [2]:
from datasets import load_from_disk
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch
import gc
from peft import PeftModel
from langchain_core.prompts import ChatPromptTemplate, HumanMessagePromptTemplate, SystemMessagePromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
import dotenv
from langchain_openai import ChatOpenAI
import os

/usr/local/lib/python3.10/dist-packages/transformers/utils/hub.py:128: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [3]:
dotenv.load_dotenv()

True

In [4]:
def clean_memory():
    for var in ['foundation_model', 'tokenizer']:
        if var in globals():
            del globals()[var]

    if torch.cuda.is_available():
        torch.cuda.ipc_collect()
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.empty_cache()

    gc.collect()
    print('Memory is cleaned')

def print_memory():
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated(0) / 1024**3
        reserved = torch.cuda.memory_reserved(0) / 1024**3
        print(f'VRAM allocated {allocated}gb, reserved {reserved}gb')
    else:
        print('No cuda')


In [5]:
clean_memory()
print_memory()

Memory is cleaned
VRAM allocated 0.0gb, reserved 0.0gb


In [6]:
test_dataset = load_from_disk('test_dataset')

In [7]:
test_dataset

Dataset({
    features: ['conversations', 'source', 'score', 'openai_dialog', 'text'],
    num_rows: 198
})

In [8]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

In [9]:
model_name = 'mistralai/Mistral-7B-Instruct-v0.3'
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

# гарантируем eos_token_id
if tokenizer.eos_token_id is None and tokenizer.eos_token is not None:
    tokenizer.eos_token_id = tokenizer.convert_tokens_to_ids(tokenizer.eos_token)

foundation_model = AutoModelForCausalLM.from_pretrained(model_name,
                                                        quantization_config=bnb_config,
                                                        device_map="auto",
                                                        attn_implementation="flash_attention_2")

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [10]:
print_memory()

VRAM allocated 3.85457706451416gb, reserved 3.859375gb


In [11]:
def apply_model(model, row):
    chat = []
    for i in row['openai_dialog']:
        if i['role'] == 'user':
            chat.append(i)
            break

    prompt = tokenizer.apply_chat_template(
        chat,
        add_generation_prompt=True,
        tokenize=False,
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    gen = model.generate(**inputs,
                         max_new_tokens=1024,
                         do_sample=False,
                         return_dict_in_generate=True,
                         repetition_penalty=1.5,
                         eos_token_id=tokenizer.eos_token_id,
                         pad_token_id=tokenizer.eos_token_id,)

    prompt_len = inputs["attention_mask"].sum(dim=1).item()
    new_tokens = gen.sequences[0, prompt_len:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=False)
    return answer
        

In [12]:
def apply_foundation_model(row):
    answer = apply_model(foundation_model, row)
    return {'foundation_model_answer': answer}

In [13]:
test_dataset = test_dataset.map(apply_foundation_model)

Map:   0%|          | 0/198 [00:00<?, ? examples/s]

From v4.47 onwards, when a model cache is to be returned, `generate` will return a `Cache` instance instead by default (as opposed to the legacy tuple of tuples format). If you want to keep returning the legacy format, please set `return_legacy_cache=True`.


In [14]:
lora_model = PeftModel.from_pretrained(foundation_model, './peft_lab_outputs/lora_adapter_3')
lora_model.eval()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): MistralForCausalLM(
      (model): MistralModel(
        (embed_tokens): Embedding(32768, 4096)
        (layers): ModuleList(
          (0-31): 32 x MistralDecoderLayer(
            (self_attn): MistralFlashAttention2(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k

In [15]:
def apply_lora_model(row):
    answer = apply_model(lora_model, row)
    return {'lora_model_answer': answer}

In [ ]:
test_dataset = test_dataset.map(apply_lora_model)

Map:   0%|          | 0/198 [00:00<?, ? examples/s]

In [18]:
print(test_dataset.select([3])['openai_dialog'])

Column([[{'content': 'How did the evolution of herbivorous mammals and their adaptations help them survive in their respective habitats and compete for resources with other animals?', 'role': 'user'}, {'content': 'The evolution of herbivorous mammals and their adaptations have played a significant role in their survival in various habitats and competition for resources with other animals. These adaptations can be observed in their anatomical, physiological, and behavioral traits, which have allowed them to exploit different food sources, avoid predation, and coexist with other species. Some of the key adaptations include:\n\n1. Dental adaptations: Herbivorous mammals have evolved specialized teeth for processing plant material. For example, many herbivores have sharp incisors for cutting and tearing plant material, and flat molars for grinding and breaking down fibrous plant matter. This allows them to efficiently consume and digest a wide variety of plant-based diets.\n\n2. Digestive 

In [19]:
class JudgeAnswer(BaseModel):
    ChosenModel: int = Field(description='Model number, which is better. -1 for the first model, 1 for the second model. 0 if models are equal.')
    Reason: str = Field(description='Reason why chosen model is better.')

In [20]:
parser = PydanticOutputParser(pydantic_object=JudgeAnswer)

In [21]:
llm = ChatOpenAI(
    api_key=os.environ['API_KEY'],
    base_url=os.environ['API_BASE_URL'],
    temperature=0.0,
    model='qwen-3-32b'
)

In [22]:
def get_prompt():
    return ChatPromptTemplate.from_messages(
        [
            SystemMessagePromptTemplate.from_template("""
                You are llm judge. You must compare two models.
                You are given instruct between xml tags <instruction> and </instruction>
                You are given first model answer between xml tags <answer1> and </answer1>.
                You are given second model answer between xml tags <answer2> and </answer2>.
                You are given ideal answer between xml tags <ideal> and </ideal>

                Chose the best answer:
                - -1 if first model is better;
                - 0 if models are equeal;
                - 1 if second model is better;

                Criterias:
                - text style
                - correctness
                - faithfulness
                - precision
                - recall

                {format_instruction}
            """),
            HumanMessagePromptTemplate.from_template("""
            Judge which model is better
            <answer1>
            {foundation_model_answer}
            </answer1>
            <answer2>
            {lora_model_answer}
            </answer2>
            """)
        ]
    )

In [27]:
def llm_judge(row):
    foundation_model_answer = 'XUY' #row['foundation_model_answer']
    lora_model_answer = 'PIZDA' #row['lora_model_answer']
    judge_answer = (get_prompt() | llm | parser).invoke({'foundation_model_answer': foundation_model_answer,
                                         'lora_model_answer': lora_model_answer,
                                         #'instruction': instruction
                                         #'ideal_answer': ideal_answer,
                                         'format_instruction': parser.get_format_instructions()
                                         })
    print(judge_answer)

In [29]:
test_dataset.select([3]).map(llm_judge)

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

JUDGING
ChosenModel=0 Reason="Both answers 'XUY' and 'PIZDA' are identical in terms of text style, correctness, faithfulness, precision, and recall as they are both single, uninformative responses. Neither provides meaningful content or addresses any potential instruction, making them equally poor choices. The comparison cannot favor one over the other based on the given criteria."


Dataset({
    features: ['conversations', 'source', 'score', 'openai_dialog', 'text'],
    num_rows: 1
})